In [2]:
import pandas as pd
import re

In [3]:
df=pd.read_csv("flipkart_com-ecommerce_sample.csv")
#print(df.columns)
#rint(df.info())
#print(df.head(1))
df1=df["is_FK_Advantage_product"]
print(df1)

0        False
1        False
2        False
3        False
4        False
         ...  
19997    False
19998    False
19999    False
20000      NaN
20001      NaN
Name: is_FK_Advantage_product, Length: 20002, dtype: object


In [4]:
# noms_uniques = df['product_name'].unique()

# # On mesure la taille de la liste
# print(len(noms_uniques))
noms_uniques = df['product_name'].unique()
df_uniques = pd.DataFrame(noms_uniques, columns=['product_name_unique'])

# 3. Sauvegarder ce résultat dans un nouveau fichier CSV
# index=False évite d'ajouter une colonne de numéros inutile au début du fichier
#df_uniques.to_csv("produits_uniques.csv", index=False)

#print("Le fichier 'produits_uniques.csv' a été créé avec succès !")

In [6]:
df_important = df[["uniq_id", "product_name", "description", "product_category_tree", "brand"]]
print(df_important.isnull().sum())
df_important = df_important.fillna("")
print(df_important.isnull().sum())


uniq_id                     2
product_name                2
description                 4
product_category_tree       2
brand                    5866
dtype: int64
uniq_id                  0
product_name             0
description              0
product_category_tree    0
brand                    0
dtype: int64


In [7]:
df_important["document_text"] = (
    df_important["product_name"] + " " +
    df_important["description"] + " " +
    df_important["product_category_tree"] + " " +
    df_important["brand"]
)

In [8]:
print(df_important[["uniq_id", "product_name", "document_text"]].head())

                            uniq_id                           product_name  \
0  c2d766ca982eca8304150849735ffef9    Alisha Solid Women's Cycling Shorts   
1  7f7036a6d550aaa89d34c77bd39a5e48    FabHomeDecor Fabric Double Sofa Bed   
2  f449ec65dcbc041b6ae5e6a32717d01b                             AW Bellies   
3  0973b37acd0c664e3de26e97e5571454    Alisha Solid Women's Cycling Shorts   
4  bc940ea42ee6bef5ac7cea3fb5cfbee7  Sicons All Purpose Arnica Dog Shampoo   

                                       document_text  
0  Alisha Solid Women's Cycling Shorts Key Featur...  
1  FabHomeDecor Fabric Double Sofa Bed FabHomeDec...  
2  AW Bellies Key Features of AW Bellies Sandals ...  
3  Alisha Solid Women's Cycling Shorts Key Featur...  
4  Sicons All Purpose Arnica Dog Shampoo Specific...  


In [9]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [10]:
df_important["clean_text"] = df_important["document_text"].apply(clean_text)

In [11]:
print(df_important[["document_text", "clean_text"]].head())

                                       document_text  \
0  Alisha Solid Women's Cycling Shorts Key Featur...   
1  FabHomeDecor Fabric Double Sofa Bed FabHomeDec...   
2  AW Bellies Key Features of AW Bellies Sandals ...   
3  Alisha Solid Women's Cycling Shorts Key Featur...   
4  Sicons All Purpose Arnica Dog Shampoo Specific...   

                                          clean_text  
0  alisha solid women s cycling shorts key featur...  
1  fabhomedecor fabric double sofa bed fabhomedec...  
2  aw bellies key features of aw bellies sandals ...  
3  alisha solid women s cycling shorts key featur...  
4  sicons all purpose arnica dog shampoo specific...  


In [12]:
def tokenize(text):
    return text.split()

In [14]:
df_important["tokens"] = df_important["clean_text"].apply(tokenize)
print(df_important[["clean_text", "tokens"]].head())

                                          clean_text  \
0  alisha solid women s cycling shorts key featur...   
1  fabhomedecor fabric double sofa bed fabhomedec...   
2  aw bellies key features of aw bellies sandals ...   
3  alisha solid women s cycling shorts key featur...   
4  sicons all purpose arnica dog shampoo specific...   

                                              tokens  
0  [alisha, solid, women, s, cycling, shorts, key...  
1  [fabhomedecor, fabric, double, sofa, bed, fabh...  
2  [aw, bellies, key, features, of, aw, bellies, ...  
3  [alisha, solid, women, s, cycling, shorts, key...  
4  [sicons, all, purpose, arnica, dog, shampoo, s...  


In [15]:
stop_words = {
    "the", "and", "for", "with", "of", "in", "on", "a", "an",
    "to", "is", "are", "this", "that", "by", "from", "as", "at",
    "it", "be", "or", "your", "you"
}

In [16]:
def remove_stop_words(tokens):
    return [word for word in tokens if word not in stop_words]

In [17]:
df_important["tokens"] = df_important["tokens"].apply(remove_stop_words)

In [18]:
def remove_short_words(tokens):
    return [word for word in tokens if len(word) > 1]

In [19]:
df_important["tokens"] = df_important["tokens"].apply(remove_short_words)

In [20]:
df_important["final_text"] = df_important["tokens"].apply(lambda tokens: " ".join(tokens))

In [21]:
print(df_important[["uniq_id", "product_name", "final_text"]].head())

                            uniq_id                           product_name  \
0  c2d766ca982eca8304150849735ffef9    Alisha Solid Women's Cycling Shorts   
1  7f7036a6d550aaa89d34c77bd39a5e48    FabHomeDecor Fabric Double Sofa Bed   
2  f449ec65dcbc041b6ae5e6a32717d01b                             AW Bellies   
3  0973b37acd0c664e3de26e97e5571454    Alisha Solid Women's Cycling Shorts   
4  bc940ea42ee6bef5ac7cea3fb5cfbee7  Sicons All Purpose Arnica Dog Shampoo   

                                          final_text  
0  alisha solid women cycling shorts key features...  
1  fabhomedecor fabric double sofa bed fabhomedec...  
2  aw bellies key features aw bellies sandals wed...  
3  alisha solid women cycling shorts key features...  
4  sicons all purpose arnica dog shampoo specific...  


In [23]:
print("Nom du produit :")
print(df_important.loc[0, "product_name"])

print("\nTexte original :")
print(df_important.loc[0, "document_text"])

print("\nTexte nettoyé :")
print(df_important.loc[0, "clean_text"])

print("\nTokens :")
print(df_important.loc[0, "tokens"])

print("\nTexte final :")
print(df_important.loc[0, "final_text"])

Nom du produit :
Alisha Solid Women's Cycling Shorts

Texte original :
Alisha Solid Women's Cycling Shorts Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Navy, Red, Navy,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 3 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTHT_3P_21 In the Box 3 shorts ["Clothing >> Women's Clothing >> Lingerie, Sleep & Swimwear >> Shorts >> Alisha Shorts >> Alisha Solid Women's Cycling Shorts"] Alisha

Texte nettoyé :
alisha solid women s cycling shorts key features of alisha solid women s cycling shorts cotton lycra navy red navy specifications of alisha solid women s cycling shorts shorts details number of contents in sales package pack of 3 fabric cotton lycra type cycling shorts general details pattern solid ideal for women s fabric care

In [24]:
df_important.to_csv("flipkart_prepared.csv", index=False)